# Градиентный бустинг своими руками

**Внимание:** в тексте задания произошли изменения - поменялось число деревьев (теперь 50), правило изменения величины шага в задании 3 и добавился параметр `random_state` у решающего дерева. Правильные ответы не поменялись, но теперь их проще получить. Также исправлена опечатка в функции `gbm_predict`.

В этом задании будет использоваться датасет `boston` из `sklearn.datasets`. Оставьте последние 25% объектов для контроля качества, разделив `X` и `y` на `X_train`, `y_train` и `X_test`, `y_test`.

Целью задания будет реализовать простой вариант градиентного бустинга над регрессионными деревьями для случая квадратичной функции потерь.

In [ ]:
from sklearn import ensemble, model_selection, metrics, datasets, tree, linear_model

import numpy as np
import pandas as pd
import xgboost as xgb

In [ ]:
%pylab inline

In [ ]:
boston = datasets.load_boston()
boston

In [ ]:
X = boston.data
X

In [ ]:
y = boston.target
y

In [ ]:
len(X), len(y), len(boston.feature_names)

In [7]:
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size = 0.25, shuffle=False)

In [8]:
len(X_train), len(X_test), len(y_train), len(y_test)

(379, 127, 379, 127)

## Задание 1

Как вы уже знаете из лекций, **бустинг** - это метод построения композиций базовых алгоритмов с помощью последовательного добавления к текущей композиции нового алгоритма с некоторым коэффициентом. 

Градиентный бустинг обучает каждый новый алгоритм так, чтобы он приближал антиградиент ошибки по ответам композиции на обучающей выборке. Аналогично минимизации функций методом градиентного спуска, в градиентном бустинге мы подправляем композицию, изменяя алгоритм в направлении антиградиента ошибки.

Воспользуйтесь формулой из лекций, задающей ответы на обучающей выборке, на которые нужно обучать новый алгоритм (фактически это лишь чуть более подробно расписанный градиент от ошибки), и получите частный ее случай, если функция потерь `L` - квадрат отклонения ответа композиции `a(x)` от правильного ответа `y` на данном `x`.

Если вы давно не считали производную самостоятельно, вам поможет таблица производных элементарных функций (которую несложно найти в интернете) и правило дифференцирования сложной функции. После дифференцирования квадрата у вас возникнет множитель 2 — т.к. нам все равно предстоит выбирать коэффициент, с которым будет добавлен новый базовый алгоритм, проигноируйте этот множитель при дальнейшем построении алгоритма.

## Задание 2

Заведите массив для объектов `DecisionTreeRegressor` (будем их использовать в качестве базовых алгоритмов) и для вещественных чисел (это будут коэффициенты перед базовыми алгоритмами). 

В цикле от обучите последовательно 50 решающих деревьев с параметрами `max_depth=5` и `random_state=42` (остальные параметры - по умолчанию). В бустинге зачастую используются сотни и тысячи деревьев, но мы ограничимся 50, чтобы алгоритм работал быстрее, и его было проще отлаживать (т.к. цель задания разобраться, как работает метод). Каждое дерево должно обучаться на одном и том же множестве объектов, но ответы, которые учится прогнозировать дерево, будут меняться в соответствие с полученным в задании 1 правилом. 

Попробуйте для начала всегда брать коэффициент равным 0.9. Обычно оправдано выбирать коэффициент значительно меньшим - порядка 0.05 или 0.1, но т.к. в нашем учебном примере на стандартном датасете будет всего 50 деревьев, возьмем для начала шаг побольше.

В процессе реализации обучения вам потребуется функция, которая будет вычислять прогноз построенной на данный момент композиции деревьев на выборке `X`:

```
def gbm_predict(X):
    return [sum([coeff * algo.predict([x])[0] for algo, coeff in zip(base_algorithms_list, coefficients_list)]) for x in X]
(считаем, что base_algorithms_list - список с базовыми алгоритмами, coefficients_list - список с коэффициентами перед алгоритмами)
```

Эта же функция поможет вам получить прогноз на контрольной выборке и оценить качество работы вашего алгоритма с помощью `mean_squared_error` в `sklearn.metrics`. 

Возведите результат в степень 0.5, чтобы получить `RMSE`. Полученное значение `RMSE` — **ответ в пункте 2**.

In [9]:
# массив для алгоритмов
base_algorithms_list = []
# массив для коэффициентов
coefficients_list = []

# функция, вычисляющая прогноз построенной на данный момент композиции деревьев на выборке X
def gbm_predict(X):
    return [sum([coeff * algo.predict([x])[0] for algo, coeff in zip(base_algorithms_list, coefficients_list)]) for x in X]

# выборка для начального алгоритма
y_train_cur = y_train 

# градиентный бустинг
for i in range(50): 
    # строим очередной алгоритм
    alg = tree.DecisionTreeRegressor(max_depth=5, random_state=42) 
    alg.fit(X_train, y_train_cur) 
    
    base_algorithms_list.append(alg) 
    coefficients_list.append(0.9) 

    # выборка для следующего алгоритма
    y_train_cur = y_train - gbm_predict(X_train)

In [10]:
# получаем прогноз на контрольной выборке
mse = metrics.mean_squared_error(y_test, gbm_predict(X_test)) 
mse

29.763826724740767

In [11]:
# корень из среднеквадратичной ошибки (MSE)
mse ** 0.5

5.455623403859614

In [12]:
with open("grad_boosting_answer2.txt", "w") as fout:
        fout.write(str(mse ** 0.5))

## Задание 3

Вас может также беспокоить, что двигаясь с постоянным шагом, вблизи минимума ошибки ответы на обучающей выборке меняются слишком резко, перескакивая через минимум. 

Попробуйте уменьшать вес перед каждым алгоритмом с каждой следующей итерацией по формуле `0.9 / (1.0 + i)`, где `i` - номер итерации (от 0 до 49). Используйте качество работы алгоритма как **ответ в пункте 3**. 

В реальности часто применяется следующая стратегия выбора шага: как только выбран алгоритм, подберем коэффициент перед ним численным методом оптимизации таким образом, чтобы отклонение от правильных ответов было минимальным. Мы не будем предлагать вам реализовать это для выполнения задания, но рекомендуем попробовать разобраться с такой стратегией и реализовать ее при случае для себя.

In [13]:
# массив для алгоритмов
base_algorithms_list = []
# массив для коэффициентов
coefficients_list = []

# выборка для начального алгоритма
y_train_cur = y_train 

# градиентный бустинг
for i in range(50): 
    # строим очередной алгоритм
    alg = tree.DecisionTreeRegressor(max_depth=5, random_state=42) 
    alg.fit(X_train, y_train_cur) 
    
    base_algorithms_list.append(alg) 
    coefficients_list.append(0.9 / (1.0 + i)) 

    # выборка для следующего алгоритма
    y_train_cur = y_train - gbm_predict(X_train)
    
# получаем прогноз на контрольной выборке
mse = metrics.mean_squared_error(y_test, gbm_predict(X_test)) 
mse ** 0.5

4.812550945781194

In [14]:
with open("grad_boosting_answer3.txt", "w") as fout:
        fout.write(str(mse ** 0.5))

## Задание 4

Реализованный вами метод - градиентный бустинг над деревьями - очень популярен в машинном обучении. Он представлен как в самой библиотеке `sklearn`, так и в сторонней библиотеке `XGBoost`, которая имеет свой питоновский интерфейс. На практике `XGBoost` работает заметно лучше `GradientBoostingRegressor` из `sklearn`, но для этого задания вы можете использовать любую реализацию. 

Исследуйте, переобучается ли градиентный бустинг с ростом числа итераций (и подумайте, почему), а также с ростом глубины деревьев. На основе наблюдений выпишите через пробел номера правильных из приведенных ниже утверждений в порядке возрастания номера (это будет **ответ в п.4**):

    1. С увеличением числа деревьев, начиная с некоторого момента, качество работы градиентного бустинга не меняется существенно.

    2. С увеличением числа деревьев, начиная с некоторого момента, градиентный бустинг начинает переобучаться.

    3. С ростом глубины деревьев, начиная с некоторого момента, качество работы градиентного бустинга на тестовой выборке начинает ухудшаться.

    4. С ростом глубины деревьев, начиная с некоторого момента, качество работы градиентного бустинга перестает существенно изменяться

#### зависимость от числа деревьев

In [ ]:
# массив для результатов
tree_num_list = []

tree_num_param = np.arange(1, 200, 1)
for i in tree_num_param:
    alg = ensemble.GradientBoostingRegressor(n_estimators=i, max_depth=5)
    alg.fit(X_train, y_train_cur) 
    
    # получаем прогноз на контрольной выборке
    mse = metrics.mean_squared_error(y_test, alg.predict(X_test)) 
    tree_num_list.append(mse ** 0.5)

In [ ]:
from scipy.signal import savgol_filter
pylab.grid(True)
pylab.plot(tree_num_param, tree_num_list, 'g-')
pylab.plot(tree_num_param, savgol_filter(tree_num_list, len(tree_num_param), 7), color='red')
pylab.title('зависимость RMSE градиентного бустинга от кол-ва деревьев')

#### зависимость от глубины деревьев

In [ ]:
# массив для результатов
tree_deep_list = []

tree_deep_param = np.arange(1, 50, 1)
for i in tree_deep_param:
    alg = ensemble.GradientBoostingRegressor(n_estimators=50, max_depth=i)
    alg.fit(X_train, y_train_cur) 
    
    # получаем прогноз на контрольной выборке
    mse = metrics.mean_squared_error(y_test, alg.predict(X_test)) 
    tree_deep_list.append(mse ** 0.5)

In [ ]:
pylab.grid(True)
pylab.plot(tree_deep_param, tree_deep_list, 'b-')
pylab.plot(tree_deep_param, savgol_filter(tree_deep_list, len(tree_deep_param), 7), color='red')
pylab.title('зависимость RMSE градиентного бустинга от глубины деревьев')

1. С увеличением числа деревьев, начиная с некоторого момента, качество работы градиентного бустинга не меняется существенно.

2. С увеличением числа деревьев, начиная с некоторого момента, градиентный бустинг начинает переобучаться.

3. С ростом глубины деревьев, начиная с некоторого момента, качество работы градиентного бустинга на тестовой выборке начинает ухудшаться.

4. С ростом глубины деревьев, начиная с некоторого момента, качество работы градиентного бустинга перестает существенно изменяться

In [ ]:
with open("grad_boosting_answer4.txt", "w") as fout:
        fout.write('2 3')

## Задание 5

Сравните получаемое с помощью градиентного бустинга качество с качеством работы линейной регрессии. 

Для этого обучите `LinearRegression` из `sklearn.linear_model` (с параметрами по умолчанию) на обучающей выборке и оцените для прогнозов полученного алгоритма на тестовой выборке `RMSE`. Полученное качество - ответ в **пункте 5**. 

В данном примере качество работы простой модели должно было оказаться хуже, но не стоит забывать, что так бывает не всегда. В заданиях к этому курсу вы еще встретите пример обратной ситуации.

In [15]:
alg = linear_model.LinearRegression()
alg.fit(X_train, y_train) 

mse = metrics.mean_squared_error(y_test, alg.predict(X_test)) 
mse ** 0.5

8.254979753549156

In [16]:
with open("grad_boosting_answer5.txt", "w") as fout:
    fout.write(str(mse ** 0.5))